In [1]:
import json
from collections import Counter
from dotenv import load_dotenv
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_qdrant import QdrantVectorStore
from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate
from langchain_qdrant import FastEmbedSparse, RetrievalMode
from qdrant_client import QdrantClient, models
from sentence_transformers import CrossEncoder

## 1. Creating LangChain Documents

In [2]:
def load_rules(path):
    """Load the rule cards created in notebook 01."""
    with open(path) as f:
        return json.load(f)

def rules_to_documents(rules):
    """Turn each rule card into a LangChain Document."""
    output = []
    for rule in rules:
        output.append(
            Document(
                page_content=rule["text"],
                metadata={
                    "rule_id": rule["rule_id"],
                    "title": rule["title"],
                    "updated": rule["updated"],
                    "chapter": rule["chapter"],
                    "start_page": rule["start_page"],
                    "end_page": rule["end_page"],
                    "printed_start": rule["printed_start"],
                    "printed_end": rule["printed_end"],
                }
            )

        )

    return output

rules = load_rules("../data/processed/b3_rules.json")
documents = rules_to_documents(rules)

print(len(documents))
print(documents[0].metadata)
print(documents[0].page_content[:200])

123
{'rule_id': 'B3-1-01', 'title': 'Comprehensive Risk Assessment', 'updated': '11/05/2025', 'chapter': 'B3-1', 'start_page': 286, 'end_page': 288, 'printed_start': 268, 'printed_end': 269}
B3-1-01, Comprehensive Risk Assessment (11/05/2025)
Introduction
This topic contains information on the comprehensive risk assessment approach to underwriting, including:
Overview
Comprehensive Risk A


## 2. Chunking

In [3]:
def split_documents(documents, chunk_size=1500, chunk_overlap=200):
    """Split long documents into chunks."""
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap
    )
    return splitter.split_documents(documents)

chunks = split_documents(documents)

print("Total chunks:", len(chunks))

per_rule = Counter(c.metadata["rule_id"] for c in chunks)
print("Rules kept whole:", sum(1 for n in per_rule.values() if n == 1))
print("Most-split rules:", per_rule.most_common(3))

Total chunks: 401
Rules kept whole: 34
Most-split rules: [('B3-5.3-09', 18), ('B3-2-02', 15), ('B3-6-05', 13)]


In [4]:
dti_chunks = [c for c in chunks if c.metadata["rule_id"] == "B3-6-02"]
for i, c in enumerate(dti_chunks):
    print(f"\n--- Chunk {i} ({len(c.page_content)} chars) ---")
    print(c.page_content[:150])


--- Chunk 0 (1478 chars) ---
B3-6-02, Debt-to-Income Ratios (04/02/2025)
Introduction
This topic contains information on the use of the debt-to-income (DTI) ratio, including:
DTI 

--- Chunk 1 (1482 chars) ---
including:
cash-out refinance transactions — the maximum ratio may be lower for loan casefiles underwritten
through DU (see B2-1.3-03, Cash-Out Refina

--- Chunk 2 (1488 chars) ---
subordinate lien payments) or rental payments (see B3-6-05, Monthly Debt Obligations);
if the subject loan is a second home or investment property, us

--- Chunk 3 (1416 chars) ---
when qualifying borrowers. This is acceptable as long as Fannie Mae’s minimum requirements are met,
and lenders consistently apply the same approach t

--- Chunk 4 (1451 chars) ---
In all cases, if the lender determines that there is new subordinate financing on the subject property during the
loan process, the mortgage loan must

--- Chunk 5 (1462 chars) ---
recalculated outside of DU.
4
• If the recalculated DTI ratio exc

In [5]:
def add_chunk_headers(chunks):
    """Return new chunks with a context header and a chunk_index"""
    position = Counter()
    output = []
    for chunk in chunks:
        rule_id = chunk.metadata["rule_id"]
        header = f"Fannie Mae Selling Guide, {rule_id}: {chunk.metadata['title']}\n\n"
        output.append(
            Document(
                page_content=header + chunk.page_content,
                metadata={**chunk.metadata, "chunk_index": position[rule_id]},
            )
        )
        position[rule_id] += 1
    return output

chunks_with_headers = add_chunk_headers(chunks)

dti = [c for c in chunks_with_headers if c.metadata["rule_id"] == "B3-6-02"]
print("\nChunk 1 metadata:", dti[1].metadata)
print("\nChunk 1 text:\n" + dti[1].page_content[:250])


Chunk 1 metadata: {'rule_id': 'B3-6-02', 'title': 'Debt-to-Income Ratios', 'updated': '04/02/2025', 'chapter': 'B3-6', 'start_page': 524, 'end_page': 528, 'printed_start': 506, 'printed_end': 509, 'chunk_index': 1}

Chunk 1 text:
Fannie Mae Selling Guide, B3-6-02: Debt-to-Income Ratios

including:
cash-out refinance transactions — the maximum ratio may be lower for loan casefiles underwritten
through DU (see B2-1.3-03, Cash-Out Refinance Transactions);
high LTV refinance tran


## 3. Models (Dense Embedding, Sparse Embedding, Reranker)

In [6]:
# Embedding model (Used for Semantic Search)
embedding_model = HuggingFaceEmbeddings(
    model_name="BAAI/bge-small-en-v1.5",
    encode_kwargs={"normalize_embeddings": True},
)

# Sparse Model (Used for Keyword Search (BM25)) 
sparse_model = FastEmbedSparse(model_name="Qdrant/bm25")

# Reranker
reranker = CrossEncoder("BAAI/bge-reranker-base")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

## 4. Vector Database

In [7]:
COLLECTION = "selling_guide_b3"
PATH = "../data/qdrant"

def build_vector_store(chunks, embedding, sparse_embedding, path, collection_name):
    """Embed chunks into a Qdrant collection with dense + BM25 vectors (hybrid search)."""
    client = QdrantClient(path=path)

    # Start fresh on every run
    if client.collection_exists(collection_name):
        client.delete_collection(collection_name)

    dense_size = len(embedding.embed_query("test"))   # 384 for bge-small

    client.create_collection(
        collection_name=collection_name,
        vectors_config=models.VectorParams(size=dense_size, distance=models.Distance.COSINE),
        sparse_vectors_config={
            "langchain-sparse": models.SparseVectorParams(modifier=models.Modifier.IDF)
        },
    )

    store = QdrantVectorStore(
        client=client,
        collection_name=collection_name,
        embedding=embedding,
        sparse_embedding=sparse_embedding,
        retrieval_mode=RetrievalMode.HYBRID,
        sparse_vector_name="langchain-sparse",
    )

    store.add_documents(chunks)
    return store


vector_store = build_vector_store(
    chunks_with_headers, embedding_model, sparse_model, PATH, COLLECTION
)

# Baseline: the same collection, searched with semantics only
semantic_store = QdrantVectorStore(
    client=vector_store.client,
    collection_name=COLLECTION,
    embedding=embedding_model,
    sparse_embedding=sparse_model,
    retrieval_mode=RetrievalMode.DENSE,
    sparse_vector_name="langchain-sparse"
)

params = vector_store.client.get_collection(COLLECTION).config.params
print("Points stored:", vector_store.client.count(COLLECTION).count)
print("Dense vectors: ", params.vectors)
print("Sparse vectors:", params.sparse_vectors)

Points stored: 401
Dense vectors:  size=384 distance=<Distance.COSINE: 'Cosine'> hnsw_config=None quantization_config=None on_disk=None memory=None datatype=None multivector_config=None
Sparse vectors: {'langchain-sparse': SparseVectorParams(index=None, modifier=<Modifier.IDF: 'idf'>)}


## 5. Retrieval & Evaluation

In [8]:
def rerank(question, docs, top_n=5):
    """Score each doc against the question and return the top_n best."""
    scores = reranker.predict([(question, d.page_content) for d in docs])
    ranked = sorted(zip(scores, docs), key=lambda x: x[0], reverse=True)
    return [d for _, d in ranked[:top_n]]


def retrieve(question, k_candidates=20, top_n=5):
    """Two-stage retrieval: hybrid search for candidates, then rerank to the best few."""
    candidates = vector_store.similarity_search(question, k=k_candidates)
    return rerank(question, candidates, top_n=top_n)

In [9]:
q = "Can my parents help me with the down payment?"
print("Hybrid only:  ", [d.metadata["rule_id"] for d in vector_store.similarity_search(q, k=5)])
print("With rerank:  ", [d.metadata["rule_id"] for d in retrieve(q)])

Hybrid only:   ['B3-4.3-06', 'B3-5.4-03', 'B3-4.3-08', 'B3-6-08', 'B3-4.2-03']
With rerank:   ['B3-4.3-08', 'B3-4.3-04', 'B3-4.3-08', 'B3-4.3-06', 'B3-4.3-01']


In [10]:
eval_set = [
    {"question": "What is the maximum debt-to-income ratio for a manually underwritten loan?", "expected": ["B3-6-02"]},
    {"question": "How long must a borrower wait after a bankruptcy?", "expected": ["B3-5.3-07"]},
    {"question": "Can my parents help me with the down payment?", "expected": ["B3-4.3-04"]},
    {"question": "What is the minimum credit score for a fixed-rate loan?", "expected": ["B3-5.1-01"]},
    {"question": "What counts as a large deposit in a bank statement?", "expected": ["B3-4.2-02"]},
    {"question": "When is a borrower considered self-employed?", "expected": ["B3-3.5-01"]},
    {"question": "Can I use cryptocurrency for my down payment?", "expected": ["B3-4.1-04"]},
    {"question": "How do I calculate monthly income if I'm paid every two weeks?", "expected": ["B3-3.3-01"]},
    {"question": "How many months of reserves are needed for an investment property?", "expected": ["B3-4.1-01"]},
    {"question": "What payment is used for a deferred student loan?", "expected": ["B3-6-05"]},
    {"question": "Can rental income from the home I'm moving out of help me qualify?", "expected": ["B3-3.8-05"]},
    {"question": "When must the lender verify the borrower is still employed before closing?", "expected": ["B3-3.1-04"]},
    {"question": "How long after a foreclosure can someone get a new mortgage?", "expected": ["B3-5.3-07"]},
    {"question": "Can a future pay raise be counted as income?", "expected": ["B3-3.2-02"]},
    {"question": "Can Social Security income be grossed up?", "expected": ["B3-3.4-15"]},
    {"question": "How much can the seller contribute toward closing costs?", "expected": ["B3-4.1-02"]},
    {"question": "What monthly payment is used for a credit card if none is shown on the credit report?", "expected": ["B3-6-05"]},
    {"question": "Which score is used when a borrower has three credit scores?", "expected": ["B3-5.1-02"]},
    {"question": "What documents are needed to use child support as income?", "expected": ["B3-3.4-02"]},
    {"question": "Can the seller give the buyer a gift of equity?", "expected": ["B3-4.3-05"]},
]

# Exact-term questions (form numbers, codes, acronyms)
exact_term_set = [
    {"question": "What is Form 1005 used for?", "expected": ["B3-3.2-01"]},
    {"question": "When must a loan be delivered with Special Feature Code 707?", "expected": ["B3-3.3-03"]},
    {"question": "When is SFC 343 used?", "expected": ["B3-2-09"]},
    {"question": "What is Form 1084?", "expected": ["B3-3.5-01", "B3-3.1-03"]},
    {"question": "How is the Quick Ratio calculated?", "expected": ["B3-3.6-07", "B3-3.7-01", "B3-3.7-02"]},
    {"question": "What does Confirmed CR BK EC mean?", "expected": ["B3-5.3-09"]},
    {"question": "What is IRS Form 4506-C used for?", "expected": ["B3-3.1-02"]},
    {"question": "What does MOP code 9 indicate?", "expected": ["B3-5.3-07", "B3-5.3-09"]},
    {"question": "What is Form 1006?", "expected": ["B3-4.2-01", "B3-4.2-02"]},
    {"question": "What is the income limit for an ADU?", "expected": ["B3-3.8-02"]},
]



def first_correct_rank(search_fn, question, expected, k=10):
    """Rank (1 = top) of the first result from an expected rule, or None."""
    for rank, doc in enumerate(search_fn(question, k), start=1):
        if doc.metadata["rule_id"] in expected:
            return rank
    return None


def hit_rates(search_fn, test_set, ks=(1, 3, 5)):
    """Hit@k for a search function on a test set."""
    ranks = [first_correct_rank(search_fn, t["question"], t["expected"]) for t in test_set]
    return {k: sum(1 for r in ranks if r is not None and r <= k) / len(test_set) for k in ks}


methods = {
    "Semantic":      lambda q, k: semantic_store.similarity_search(q, k=k),
    "Hybrid":        lambda q, k: vector_store.similarity_search(q, k=k),
    "Hybrid+Rerank": lambda q, k: retrieve(q, k_candidates=20, top_n=k),
}

print(f"{'Test set':<20}{'Method':<16}{'Hit@1':>7}{'Hit@3':>7}{'Hit@5':>7}")
for set_name, test_set in [("Everyday (20)", eval_set), ("Exact terms (10)", exact_term_set)]:
    for name, search_fn in methods.items():
        h = hit_rates(search_fn, test_set)
        print(f"{set_name:<20}{name:<16}{h[1]:>7.0%}{h[3]:>7.0%}{h[5]:>7.0%}")

Test set            Method            Hit@1  Hit@3  Hit@5
Everyday (20)       Semantic            85%    95%   100%
Everyday (20)       Hybrid              90%    95%    95%
Everyday (20)       Hybrid+Rerank       85%   100%   100%
Exact terms (10)    Semantic            70%    80%    90%
Exact terms (10)    Hybrid              80%   100%   100%
Exact terms (10)    Hybrid+Rerank      100%   100%   100%


## 6. LLM

In [11]:
load_dotenv()    # reads .env so the API key is available

llm = ChatGroq(
    model="openai/gpt-oss-120b",
    temperature=0,
)

response = llm.invoke("In one sentence, what is a debt-to-income ratio?")

print(response.content)
print(response.usage_metadata)

A debt‑to‑income ratio is the percentage of a person’s gross monthly income that goes toward paying all recurring debt obligations.
{'input_tokens': 83, 'output_tokens': 65, 'total_tokens': 148, 'output_token_details': {'reasoning': 30}}


## 7. RAG Pipeline

In [13]:
def format_context(docs):
    """Format chunks as numbered sources with rule ID, title, and pages."""
    blocks = []
    for i, doc in enumerate(docs, start=1):
        m = doc.metadata
        label = (
                 f"[Source {i}] {m['rule_id']}: {m['title']} "
                 f"(pages {m['printed_start']}-{m['printed_end']})"
        )
        blocks.append(f"{label}\n{doc.page_content}")
    return "\n\n".join(blocks)



SYSTEM_PROMPT = """You are an assistant that answers questions about the Fannie Mae Selling Guide, Part B3 (Underwriting Borrowers).

Rules:
- Answer ONLY using the sources provided. Do not use outside knowledge.
- Cite every fact with its rule ID and pages, like (B3-6-02, pp. 506-509).
- If the sources do not contain the answer, say: "The provided sections do not answer this question."
- Be concise and precise. Include exact numbers and conditions when the sources give them."""

prompt = ChatPromptTemplate.from_messages([
    ("system", SYSTEM_PROMPT),
    ("human", "Question: {question}\n\nSources:\n{context}"),
])

def ask(question):
    """Retrieve sources, ask the LLM, and print the answer with its sources."""
    docs = retrieve(question)
    messages = prompt.invoke({"question": question, "context": format_context(docs)})
    response = llm.invoke(messages)

    print(f"Question: {question}\n")
    print(response.content)
    print("\nRetrieved:", ", ".join(d.metadata["rule_id"] for d in docs))
    print("-" * 80)

# Test
ask("What is the maximum debt-to-income ratio for a manually underwritten loan?")
ask("Can my parents help me with the down payment?")
ask("What are the appraisal requirements for a condo?")

Question: What is the maximum debt-to-income ratio for a manually underwritten loan?

The Fannie Mae Selling Guide states that the standard ceiling for a manually underwritten loan is **36 % of the borrower’s stable monthly income**.  This limit may be stretched to **as high as 45 %** when the borrower satisfies the credit‑score and reserve requirements set out in the Eligibility Matrix (B3‑6‑02, “Maximum DTI Ratios,” pp. 506‑509).

Retrieved: B3-6-02, B3-6-02, B3-6-02, B3-6-02, B3-3.1-01
--------------------------------------------------------------------------------
Question: Can my parents help me with the down payment?

Yes. A parent may provide a **personal gift** that can be used to meet or supplement the down‑payment, closing‑costs, and reserves once the borrower’s minimum contribution has been satisfied.

**What is required**

* The gift must be documented with a **gift letter** signed by the donor (your parent). The letter must:
  * state the actual or maximum dollar amount of